In [11]:
from common.aws_config import aws_env_setup
from common.bedrock_config import get_anthropic_client
import json 

aws_env_setup()
client, model = get_anthropic_client()

In [12]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        #"temperature": temperature
        "extra_body": {"temperature": temperature}
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [13]:
def generate_dataset():
    prompt = """
        Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
        {
            "task": "Description of task",
        },
        ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
        """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages,"```json")
    text =chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [14]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URI (e.g., 's3://my-bucket-us-west-2/path/to/file'). The function should return the region code or None if not found."}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'. Include the necessary Action, Resource, and Effect fields."}, {'task': 'Write a Python function that takes an AWS CloudWatch log message and extracts the timestamp, log level (ERROR, WARN, INFO), and message content. Return these as a dictionary.'}]


In [15]:
# write file to disk
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)